# Gaze Coordination Analysis — Turn-Based Windows
**Data:** `gaze_coordination_turns.csv` — 336 MOI entries across 54 interactions  
**Windows:** Pre = last non-annotated participant turn before MOI · MOI · Post = first turn after MOI  

---
## What each method measures

| Method | Range | What it captures |
|--------|-------|------------------|
| **1a Frame-by-Frame Cosine Similarity** (`m1a_cosine_sim_mean`) | −1 to +1 | Are both participants' gaze vectors pointing in the **same direction at each moment**? +1 = identical direction every frame, 0 = orthogonal on average, −1 = opposite. |
| **1b Summary Cosine Similarity** (`m1b_summary_cosine`) | −1 to +1 | Same as 1a but compares each participant's **average gaze direction** over the whole window. Less sensitive to noise, captures the overall gaze posture rather than moment-to-moment. |
| **2 Cross-Correlation Peak** (`m2_xcorr_peak`) | −1 to +1 | Does the **magnitude** (overall intensity) of one person's gaze movements predict the other's at some time lag? Higher = stronger coupling. |
| **2 Cross-Correlation Lag** (`m2_xcorr_lag_s`) | −2 to +2s | **When** does that coupling peak? 0 = simultaneous, positive = B leads A, negative = A leads B. |
| **3 Multivariate Lagged Peak r** (`m3_mv_lag_peak_r`) | −1 to +1 | Like Method 2 but uses the full 2D gaze vector (both dimensions together) and finds the best temporal alignment. More sensitive than Method 2 for multidimensional signals. |
| **3 Multivariate Lag** (`m3_mv_lag_s`) | −2 to +2s | The lag at which Method 3's peak occurs. |

---
## What your results say

- **Methods 1a & 1b (cosine similarity): consistently positive and significant (p < 0.001)**  
  Mean frame-by-frame cosine sim ≈ 0.27–0.31 across windows — participants are gaze-aligned above chance throughout. Importantly, **pre-MOI similarity is significantly higher than during the MOI** (Wilcoxon p < 0.05), suggesting gaze alignment *drops* as the MOI occurs.

- **Method 1b (summary cosine) shows the same drop, and pre > post is also significant (p < 0.05)**  
  The overall gaze posture is most similar *before* the MOI and doesn't fully recover after.

- **Methods 2 & 3 (cross-correlation): peak xcorr is significant (p < 0.001) but multivariate peak r is only marginally significant pre-MOI**  
  Gaze movement magnitudes are coupled across all windows (~0.09–0.15). But the full 2D movement pattern (Method 3) is only weakly coupled pre-MOI (p = 0.03) and not during or after.

- **Lags are centered near 0** — gaze coupling is largely simultaneous rather than one participant clearly leading the other.

- **Valid frame counts are highly variable** (median pre = 127 frames ≈ 4.2s, median moi = 60 frames ≈ 2s) — turn-based windows vary a lot in length, which matters for interpreting the results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, ttest_1samp, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
WINDOWS       = ['pre', 'moi', 'post']
WINDOW_LABELS = {'pre': 'Pre-MOI\n(turn)', 'moi': 'MOI', 'post': 'Post-MOI\n(turn)'}
WINDOW_COLORS = {'pre': '#4878CF', 'moi': '#E24A33', 'post': '#6ACC65'}

METHODS = {
    'm1a_cosine_sim_mean': 'Method 1a: Frame-by-Frame\nCosine Similarity',
    'm1b_summary_cosine':  'Method 1b: Summary\nCosine Similarity',
    'm2_xcorr_peak':       'Method 2: Cross-Corr\nPeak',
    'm3_mv_lag_peak_r':    'Method 3: Multivariate\nLagged Peak r',
}

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
CSV_PATH = 'gaze_coordination_turns.csv'
df = pd.read_csv(CSV_PATH)

print(f'Loaded {len(df)} MOI entries from {df["interaction_id"].nunique()} interactions')
print(f'Mean MOIs per interaction: {df.groupby("interaction_id").size().mean():.1f}')
print(f'Columns: {list(df.columns)}')

# Long format — one row per MOI per window
ID_COLS = ['interaction_id', 'annotated_participant', 'non_annotated_participant',
           'event_speaker', 'participant_a', 'participant_b',
           'start_moi', 'end_moi', 'moi_duration', 'pre_overlap', 'post_overlap']

METRIC_NAMES = [
    'm1a_cosine_sim_mean', 'm1a_n_valid',
    'm1b_summary_cosine',
    'm2_xcorr_peak', 'm2_xcorr_lag_s', 'm2_n_valid',
    'm3_mv_lag_peak_r', 'm3_mv_lag_s', 'm3_n_valid',
]

long_frames = []
for w in WINDOWS:
    tmp = df[ID_COLS].copy()
    for m in METRIC_NAMES:
        col = f'{w}_{m}'
        if col in df.columns:
            tmp[m] = df[col]
    tmp['window'] = w
    long_frames.append(tmp)

dfl = pd.concat(long_frames, ignore_index=True)
dfl['window'] = pd.Categorical(dfl['window'], categories=WINDOWS, ordered=True)
print('Long-format shape:', dfl.shape)

---
## 1. Descriptive Statistics by Window

In [ ]:
key_metrics = ['m1a_cosine_sim_mean', 'm1b_summary_cosine', 'm2_xcorr_peak', 'm3_mv_lag_peak_r']
summary = dfl.groupby('window')[key_metrics].agg(['mean','median','std','count']).round(4)
print('Summary statistics by window:')
summary

---
## 2. Are all methods significantly above zero?
*Each bar = mean ± SEM. Stars = one-sample t-test vs 0.*

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
fig.suptitle('Mean Coordination by Method & Window\n(* p<0.05, ** p<0.01, *** p<0.001 vs zero)',
             fontsize=13, fontweight='bold')

for ax, metric in zip(axes, METHODS.keys()):
    means, sems, sigs = [], [], []
    for w in WINDOWS:
        vals = dfl.loc[dfl['window'] == w, metric].dropna()
        means.append(vals.mean())
        sems.append(vals.sem())
        t, p = ttest_1samp(vals, 0)
        sigs.append('***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns')))

    x = np.arange(3)
    bars = ax.bar(x, means, yerr=sems,
                  color=[WINDOW_COLORS[w] for w in WINDOWS],
                  capsize=5, edgecolor='black', alpha=0.85, width=0.55)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)

    for i, (m, s, sig) in enumerate(zip(means, sems, sigs)):
        offset = abs(s) + 0.01
        ax.text(i, m + offset if m >= 0 else m - offset - 0.03,
                sig, ha='center', fontsize=12, fontweight='bold',
                va='bottom' if m >= 0 else 'top')

    ax.set_xticks(x)
    ax.set_xticklabels([WINDOW_LABELS[w] for w in WINDOWS], fontsize=9)
    ax.set_title(METHODS[metric], fontsize=10)
    ax.set_ylabel('Mean value')

plt.tight_layout()
plt.savefig('gc_fig1_methods_vs_zero.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Pre → MOI → Post Trajectories
*The key finding: cosine similarity drops during the MOI compared to before.*

In [ ]:
int_means = dfl.groupby(['interaction_id', 'window'])[key_metrics].mean().reset_index()

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle('Pre → MOI → Post Trajectories (interaction-level means)\n'
             'Gray lines = individual interactions, brackets = Wilcoxon signed-rank',
             fontsize=13, fontweight='bold')

for ax, metric in zip(axes, METHODS.keys()):
    pivot = (
        int_means.pivot(index='interaction_id', columns='window', values=metric)
        [WINDOWS].dropna()
    )
    x = [0, 1, 2]

    for _, row in pivot.iterrows():
        ax.plot(x, row[WINDOWS].values, color='gray', alpha=0.15, linewidth=0.8)

    means = pivot.mean()
    sems  = pivot.sem()
    ax.errorbar(x, means[WINDOWS], yerr=sems[WINDOWS],
                fmt='o-', linewidth=2.5, markersize=9,
                color='black', zorder=5, capsize=5)
    for xi, w in zip(x, WINDOWS):
        ax.scatter(xi, means[w], color=WINDOW_COLORS[w], s=130, zorder=6)

    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

    # Wilcoxon brackets
    yrange  = pivot.values.max() - pivot.values.min()
    bracket_y = pivot.values.max() + yrange * 0.08
    step      = yrange * 0.08
    for k, (w1, w2, xi1, xi2) in enumerate([('pre','moi',0,1), ('moi','post',1,2), ('pre','post',0,2)]):
        common = pivot[w1].index.intersection(pivot[w2].index)
        _, p = wilcoxon(pivot.loc[common, w1], pivot.loc[common, w2])
        sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else f'ns'))
        y = bracket_y + k * step
        ax.annotate('', xy=(xi2, y), xytext=(xi1, y),
                    arrowprops=dict(arrowstyle='-', color='dimgray', lw=1.2))
        ax.text((xi1+xi2)/2, y + step*0.25, sig, ha='center', fontsize=10,
                color='black' if sig != 'ns' else 'gray')

    ax.set_xticks(x)
    ax.set_xticklabels([WINDOW_LABELS[w] for w in WINDOWS], fontsize=9)
    ax.set_title(METHODS[metric], fontsize=10)
    ax.set_ylabel('Mean value')

plt.tight_layout()
plt.savefig('gc_fig2_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Distribution of Each Method (Violin + Strip)
*Full distribution across all MOI entries, not just means.*

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
fig.suptitle('Distribution of Coordination Metrics by Window',
             fontsize=13, fontweight='bold')

for ax, metric in zip(axes, METHODS.keys()):
    data = [dfl.loc[dfl['window'] == w, metric].dropna() for w in WINDOWS]

    parts = ax.violinplot(data, positions=[0,1,2], showmedians=True, showextrema=False)
    for pc, w in zip(parts['bodies'], WINDOWS):
        pc.set_facecolor(WINDOW_COLORS[w])
        pc.set_alpha(0.65)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)

    for i, (w, vals) in enumerate(zip(WINDOWS, data)):
        jitter = np.random.uniform(-0.07, 0.07, len(vals))
        ax.scatter(i + jitter, vals, alpha=0.2, s=10, color=WINDOW_COLORS[w], zorder=3)

    ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
    ax.set_xticks([0,1,2])
    ax.set_xticklabels([WINDOW_LABELS[w] for w in WINDOWS], fontsize=9)
    ax.set_title(METHODS[metric], fontsize=10)
    ax.set_ylabel('Value')

plt.tight_layout()
plt.savefig('gc_fig3_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Lag Distributions (Methods 2 & 3)
*Where does coupling peak in time? Centered near 0 = simultaneous coordination.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Lag at Peak Coupling (positive = B leads A, negative = A leads B)',
             fontsize=13, fontweight='bold')

for ax, lag_col, title in zip(
    axes,
    ['m2_xcorr_lag_s', 'm3_mv_lag_s'],
    ['Method 2: Cross-Correlation Lag', 'Method 3: Multivariate Lagged Correlation Lag']
):
    for w in WINDOWS:
        vals = dfl.loc[dfl['window'] == w, lag_col].dropna()
        ax.hist(vals, bins=40, alpha=0.5,
                label=f"{WINDOW_LABELS[w].replace(chr(10),' ')} (n={len(vals)})",
                color=WINDOW_COLORS[w], edgecolor='white')

    ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Lag = 0 (simultaneous)')
    ax.set_title(title)
    ax.set_xlabel('Lag (seconds)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

    print(f'\n{title} — median lags:')
    for w in WINDOWS:
        vals = dfl.loc[dfl['window'] == w, lag_col].dropna()
        print(f'  {w}: median={vals.median():.3f}s, mean={vals.mean():.3f}s')

plt.tight_layout()
plt.savefig('gc_fig4_lags.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Effect of Turn Overlap on MOI Coordination
*Does it matter if the pre-turn was still ongoing when the MOI started?*

In [ ]:
moi_df = dfl[dfl['window'] == 'moi'].copy()
moi_df['pre_overlap_label'] = moi_df['pre_overlap'].map(
    {True: 'Pre-turn overlaps MOI', False: 'Pre-turn ends before MOI'}
)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('MOI Coordination by Pre-Turn Overlap', fontsize=13, fontweight='bold')

for ax, metric in zip(axes, METHODS.keys()):
    sub = moi_df[['pre_overlap_label', metric]].dropna()
    g1  = sub.loc[sub['pre_overlap_label'] == 'Pre-turn overlaps MOI', metric]
    g2  = sub.loc[sub['pre_overlap_label'] == 'Pre-turn ends before MOI', metric]
    if len(g1) > 2 and len(g2) > 2:
        _, p = mannwhitneyu(g1, g2, alternative='two-sided')
        sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else f'ns (p={p:.3f})'))
    else:
        sig = 'n/a'

    sns.boxplot(data=sub, x='pre_overlap_label', y=metric, ax=ax,
                palette={'Pre-turn overlaps MOI': '#E24A33',
                         'Pre-turn ends before MOI': '#4878CF'},
                width=0.45)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ymax = sub[metric].quantile(0.95)
    ymin = sub[metric].quantile(0.05)
    ax.text(0.5, 1.04, f'Mann-Whitney: {sig}',
            ha='center', transform=ax.transAxes, fontsize=10)
    ax.set_title(METHODS[metric], fontsize=10)
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right', fontsize=8)
    ax.set_ylabel('Value')

plt.tight_layout()
plt.savefig('gc_fig5_overlap_effect.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Valid Frames per Window
*Turn windows are variable-length — this shows how many frames were actually analyzed.*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Valid Frames per Window (30 fps — shows effective window length)',
             fontsize=13, fontweight='bold')

for ax, w in zip(axes, WINDOWS):
    col  = f'{w}_m1a_n_valid'
    vals = df[col]
    ax.hist(vals, bins=40, color=WINDOW_COLORS[w], edgecolor='white', alpha=0.85)
    ax.axvline(vals.median(), color='black', linestyle='--', linewidth=1.5,
               label=f'Median = {vals.median():.0f} frames ({vals.median()/30:.1f}s)')
    ax.set_title(WINDOW_LABELS[w])
    ax.set_xlabel('Valid frames')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    # Add secondary x-axis in seconds
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim()[0]/30, ax.get_xlim()[1]/30)
    ax2.set_xlabel('Seconds')

plt.tight_layout()
plt.savefig('gc_fig6_valid_frames.png', dpi=150, bbox_inches='tight')
plt.show()

for w in WINDOWS:
    col = f'{w}_m1a_n_valid'
    vals = df[col]
    print(f'{w}: median={vals.median():.0f} frames ({vals.median()/30:.1f}s), '
          f'mean={vals.mean():.0f} ({vals.mean()/30:.1f}s), '
          f'max={vals.max():.0f} ({vals.max()/30:.1f}s)')

---
## 8. Per-Interaction Heatmap
*Green = higher coordination, red = lower. Sorted by MOI Method 1a.*

In [ ]:
heat_cols = [
    'pre_m1a_cosine_sim_mean',  'moi_m1a_cosine_sim_mean',  'post_m1a_cosine_sim_mean',
    'pre_m1b_summary_cosine',   'moi_m1b_summary_cosine',   'post_m1b_summary_cosine',
    'pre_m2_xcorr_peak',        'moi_m2_xcorr_peak',        'post_m2_xcorr_peak',
    'pre_m3_mv_lag_peak_r',     'moi_m3_mv_lag_peak_r',     'post_m3_mv_lag_peak_r',
]
col_labels = [
    'Pre\n1a Cos', 'MOI\n1a Cos', 'Post\n1a Cos',
    'Pre\n1b Sum', 'MOI\n1b Sum', 'Post\n1b Sum',
    'Pre\n2 XC',  'MOI\n2 XC',  'Post\n2 XC',
    'Pre\n3 MV',  'MOI\n3 MV',  'Post\n3 MV',
]
heat_df = (
    df.groupby('interaction_id')[heat_cols].mean()
    .sort_values('moi_m1a_cosine_sim_mean', ascending=False)
)

fig, ax = plt.subplots(figsize=(15, max(7, len(heat_df) * 0.30)))
sns.heatmap(heat_df, ax=ax, cmap='RdYlGn', center=0,
            linewidths=0.3, linecolor='white',
            xticklabels=col_labels,
            cbar_kws={'shrink': 0.5, 'label': 'Coordination value'})
ax.set_title('Per-Interaction Mean Gaze Coordination — All Methods\n'
             '(sorted by MOI Frame-by-Frame Cosine Similarity)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Interaction ID')
plt.xticks(rotation=0, ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('gc_fig7_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 interactions (highest MOI cosine similarity):')
print(heat_df['moi_m1a_cosine_sim_mean'].head(5).round(4).to_string())
print('\nBottom 5:')
print(heat_df['moi_m1a_cosine_sim_mean'].tail(5).round(4).to_string())

---
## 9. Method Consistency: Do all 4 methods agree?

In [ ]:
# Correlation between methods within the MOI window
moi_corr_cols = ['moi_m1a_cosine_sim_mean', 'moi_m1b_summary_cosine',
                 'moi_m2_xcorr_peak', 'moi_m3_mv_lag_peak_r']
corr_labels   = ['1a Frame\nCosine', '1b Summary\nCosine',
                 '2 XCorr\nPeak', '3 MV Lag\nPeak r']

corr_df = df[moi_corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, ax=ax, mask=mask, cmap='coolwarm', center=0,
            vmin=-1, vmax=1, annot=True, fmt='.2f', linewidths=0.5,
            xticklabels=corr_labels, yticklabels=corr_labels,
            cbar_kws={'shrink': 0.7})
ax.set_title('Method Consistency — MOI Window\n'
             '(do methods agree with each other?)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('gc_fig8_method_consistency.png', dpi=150, bbox_inches='tight')
plt.show()
print('High correlation between methods = convergent validity')
print('Low correlation = methods capturing different aspects of coordination')

---
## 10. Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Gaze Coordination — Summary Dashboard (Turn-Based Windows)',
             fontsize=14, fontweight='bold', y=1.01)

# Row 1: boxplots
for ax, metric in zip(axes[0], METHODS.keys()):
    data = [dfl.loc[dfl['window'] == w, metric].dropna() for w in WINDOWS]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    for patch, w in zip(bp['boxes'], WINDOWS):
        patch.set_facecolor(WINDOW_COLORS[w])
        patch.set_alpha(0.75)
    ax.set_xticks([1,2,3])
    ax.set_xticklabels([WINDOW_LABELS[w] for w in WINDOWS], fontsize=8)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(METHODS[metric], fontsize=10)

# Row 2: KDE
for ax, metric in zip(axes[1], METHODS.keys()):
    for w in WINDOWS:
        vals = dfl.loc[dfl['window'] == w, metric].dropna()
        if len(vals) > 2:
            vals.plot.kde(ax=ax, label=WINDOW_LABELS[w].replace('\n', ' '),
                          color=WINDOW_COLORS[w], linewidth=2)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.legend(fontsize=8)
    ax.set_title(f'{METHODS[metric]} (KDE)', fontsize=10)

patches = [mpatches.Patch(color=WINDOW_COLORS[w],
                           label=WINDOW_LABELS[w].replace('\n', ' ')) for w in WINDOWS]
fig.legend(handles=patches, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), fontsize=11)
plt.tight_layout()
plt.savefig('gc_fig9_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('All figures saved with prefix gc_')